# Raptor v5 + RadImageNet residual fusion

Both unfrozen backbones receive the same preprocessed 384×384 adjacent-slice
triplet. CoAtNet S2 uses `Z + F1(X,Z)`; S4 uses `V + F2(V,Y)`.
Both MLP corrections initialize to exactly zero. The original Raptor MIL head
continues after CoAtNet pooling. This notebook builds the architecture and
optimizer and a series-aware loader. Use this loader to train it; anonymous
concatenated slice caches cannot enforce MRI series boundaries.

Attach `dreaddevelopment/raptor-knee-maxspan`,
`marwanmath/resnet-50-radimagenet-marwan`, and a `timm-1.0.22` wheel if your
runtime does not already have that version. Attach the existing training
competition series and labels for training. No checkpoint or competition data
downloads occur. Dense eval preserves the parent's per-study triplet count
(usually 94), using up to 104 sampled slices across five series.


In [ ]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

try:
    timm_version = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError:
    timm_version = None
if timm_version != '1.0.22':
    wheels = sorted(Path('/kaggle/input').rglob('timm-1.0.22-py3-none-any.whl'))
    if not wheels:
        raise RuntimeError('Attach the timm 1.0.22 wheel before running offline')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           '--no-index', '--no-deps', str(wheels[0])])


In [ ]:
"""Raptor v5 with residual RadImageNet fusion inside CoAtNet.

Input is an already preprocessed [B, K, 3, 384, 384] triplet bag. Both
encoders receive exactly the same tensor; preprocessing belongs to the loader.
"""

from __future__ import annotations

from collections.abc import Mapping
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import torch
from torch import Tensor, nn
from torch.utils.checkpoint import checkpoint


COATNET_ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
LABELS = (
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
)


@dataclass(frozen=True)
class FusionConfig:
    stage2_hidden: int = 1024
    stage4_hidden: int = 2048
    mlp_dropout: float = 0.0
    head_dropout: float = 0.2
    gradient_checkpointing: bool = False


class ConcatenationMLP(nn.Module):
    """Three affine layers with GELU, applied to each spatial position.

    Concatenation precedes every learned operation. Only the output projection
    is zero-initialized, so the residual is exactly zero without a dead MLP.
    """

    def __init__(self, first_channels: int, second_channels: int,
                 out_channels: int, hidden: int, dropout: float = 0.0):
        super().__init__()
        if min(first_channels, second_channels, out_channels, hidden) <= 0:
            raise ValueError("MLP channel dimensions must be positive")
        self.first_channels = first_channels
        self.second_channels = second_channels
        self.mlp = nn.Sequential(
            nn.LayerNorm(first_channels + second_channels),
            nn.Linear(first_channels + second_channels, hidden),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_channels),
        )
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, first: Tensor, second: Tensor) -> Tensor:
        if (first.ndim != 4 or second.ndim != 4
                or first.shape[0] != second.shape[0]
                or first.shape[2:] != second.shape[2:]
                or first.shape[1] != self.first_channels
                or second.shape[1] != self.second_channels):
            raise ValueError(f"Incompatible fusion features: {first.shape}, {second.shape}")
        joined = torch.cat((first, second), dim=1).permute(0, 2, 3, 1)
        return self.mlp(joined).permute(0, 3, 1, 2).contiguous()


class RadImageNetEncoder(nn.Module):
    """The same torchvision ResNet-50 trunk used by the baseline Rad encoder.

    Keep `backbone.0` ... `backbone.7` names for strict ResNet50.pt loading.
    conv3_x is backbone[5] (torchvision layer2); conv5_x is backbone[7]
    (layer4). No ImageNet weights are substituted for RadImageNet weights.
    """

    def __init__(self):
        super().__init__()
        from torchvision.models import resnet50

        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])

    def forward(self, image: Tensor) -> tuple[Tensor, Tensor]:
        x = image
        conv3 = None
        for index, layer in enumerate(self.backbone):
            x = layer(x)
            if index == 5:
                conv3 = x
        return conv3, x


class FusedCoAtNet(nn.Module):
    """Stage2 Z += F1(X,Z), Stage4 V += F2(V,Y), then normal CoAtNet pooling."""

    def __init__(self, coatnet: nn.Module, radimagenet: nn.Module,
                 config: FusionConfig = FusionConfig()):
        super().__init__()
        self.coatnet = coatnet
        self.radimagenet = radimagenet
        self.num_features = coatnet.num_features
        channels = [entry["num_chs"] for entry in coatnet.feature_info]
        if len(coatnet.stages) != 4 or channels[-4:] != [128, 256, 512, 1024]:
            raise ValueError("Expected the four-stage Raptor v5 CoAtNet RMLP-2")
        self.f1 = ConcatenationMLP(512, 256, 256, config.stage2_hidden, config.mlp_dropout)
        self.f2 = ConcatenationMLP(1024, 2048, 1024, config.stage4_hidden, config.mlp_dropout)
        self.gradient_checkpointing = config.gradient_checkpointing
        self.requires_grad_(True)
        self.coatnet.set_grad_checkpointing(config.gradient_checkpointing)

    def _fuse(self, mlp: nn.Module, first: Tensor, second: Tensor) -> Tensor:
        if self.gradient_checkpointing and self.training and torch.is_grad_enabled():
            return checkpoint(mlp, first, second, use_reentrant=False)
        return mlp(first, second)

    def forward_features(self, image: Tensor) -> Tensor:
        if image.ndim != 4 or tuple(image.shape[1:]) != (3, 384, 384):
            raise ValueError("Both backbones require [N, 3, 384, 384] preprocessed triplets")
        # Feed the identical image object to both trunks; no second normalization.
        x, y = self.radimagenet(image)
        z = self.coatnet.stages[0](self.coatnet.stem(image))
        z = self.coatnet.stages[1](z)  # S2 convolution, [N,256,48,48]
        z = z + self._fuse(self.f1, x, z)  # F1 input order is (X,Z)
        v = self.coatnet.stages[2](z)  # S3 receives the UPDATED Z
        v = self.coatnet.stages[3](v)  # S4 transformer, [N,1024,12,12]
        v = v + self._fuse(self.f2, v, y)  # F2 input order is (V,Y)
        return self.coatnet.norm(v)

    def forward(self, image: Tensor) -> Tensor:
        return self.coatnet.forward_head(self.forward_features(image))


class RaptorClassifier(nn.Module):
    """Original Raptor attention-MIL head, retaining checkpoint parameter names."""

    def __init__(self, backbone: nn.Module, F_dim: int, n: int = 12, drop: float = 0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x: Tensor) -> Tensor:
        if x.ndim != 5 or x.shape[0] == 0 or x.shape[1] == 0:
            raise ValueError("Raptor expects a nonempty [B,K,3,384,384] triplet bag")
        b, k = x.shape[:2]
        return self.backbone(x.flatten(0, 1)).reshape(b, k, -1)

    def head(self, feats: Tensor) -> Tensor:
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x: Tensor) -> Tensor:
        return self.head(self.encode(x))


def _checkpoint(source: str | Path | Mapping[str, Any]) -> Mapping[str, Any]:
    return source if isinstance(source, Mapping) else torch.load(
        source, map_location="cpu", weights_only=True,
    )


def build_raptor_v5_radimagenet(
    raptor_checkpoint: str | Path | Mapping[str, Any],
    radimagenet_checkpoint: str | Path | Mapping[str, Any],
    config: FusionConfig = FusionConfig(),
) -> RaptorClassifier:
    """Strict-load both pretrained parents, add zero residuals, unfreeze all.

    Raptor checkpoint: {'model': state_dict, 'arch': ..., 'res': 384}.
    Rad checkpoint: the baseline ResNet50.pt state dict, or {'state_dict': ...}.
    Load and resume a *fused* checkpoint using load_fused_checkpoint instead.
    """
    import timm

    ck = _checkpoint(raptor_checkpoint)
    if ck.get("arch", COATNET_ARCH) != COATNET_ARCH or int(ck.get("res", 384)) != 384:
        raise ValueError("Parent checkpoint must be Raptor v5 CoAtNet RMLP-2 at res=384")
    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")
    model = RaptorClassifier(coatnet, F_dim=coatnet.num_features, drop=config.head_dropout)
    model.load_state_dict(ck["model"], strict=True)
    rad = RadImageNetEncoder()
    rad_ck = _checkpoint(radimagenet_checkpoint)
    rad.load_state_dict(rad_ck.get("state_dict", rad_ck), strict=True)
    model.backbone = FusedCoAtNet(coatnet, rad, config)
    model.fusion_config = config
    model.requires_grad_(True)
    return model


def optimizer_groups(model: RaptorClassifier, backbone_lr: float = 1e-5,
                     fusion_lr: float = 1e-4, head_lr: float = 1e-4) -> list[dict[str, Any]]:
    """All parameters, exactly once, with both backbones unfrozen."""
    if not isinstance(model.backbone, FusedCoAtNet):
        raise TypeError("Expected the fused Raptor model")
    if any(not p.requires_grad for p in model.parameters()):
        raise ValueError("Both backbones and the fusion/head must remain unfrozen")
    groups = {"coatnet": [], "radimagenet": [], "fusion": [], "head": []}
    for name, parameter in model.named_parameters():
        if name.startswith("backbone.coatnet."):
            group = "coatnet"
        elif name.startswith("backbone.radimagenet."):
            group = "radimagenet"
        elif name.startswith(("backbone.f1.", "backbone.f2.")):
            group = "fusion"
        else:
            group = "head"
        groups[group].append(parameter)
    rates = dict(coatnet=backbone_lr, radimagenet=backbone_lr, fusion=fusion_lr, head=head_lr)
    return [{"name": name, "params": parameters, "lr": rates[name]}
            for name, parameters in groups.items()]


def fused_checkpoint(model: RaptorClassifier) -> dict[str, Any]:
    """Serialize model/config; trainers can add optimizer, scaler and epoch."""
    return {"format": "raptor_v5_radimagenet_v1", "arch": COATNET_ARCH, "res": 384,
            "labels": list(LABELS), "fusion_config": asdict(model.fusion_config),
            "model": model.state_dict()}


def load_fused_checkpoint(source: str | Path | Mapping[str, Any]) -> RaptorClassifier:
    import timm

    ck = _checkpoint(source)
    if (ck.get("format") != "raptor_v5_radimagenet_v1" or ck.get("arch") != COATNET_ARCH
            or ck.get("res") != 384 or ck.get("labels") != list(LABELS)):
        raise ValueError("Unsupported fused checkpoint architecture/input/label contract")
    config = FusionConfig(**ck["fusion_config"])
    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")
    backbone = FusedCoAtNet(coatnet, RadImageNetEncoder(), config)
    model = RaptorClassifier(backbone, F_dim=backbone.num_features, drop=config.head_dropout)
    model.fusion_config = config
    model.load_state_dict(ck["model"], strict=True)
    model.requires_grad_(True)
    return model


In [ ]:
"""Form Raptor triplets within SeriesInstanceUID boundaries, preserving K.

Adjacent here means adjacent in the physically ordered *sampled* series, as in
the parent Raptor loader; sampling may skip native DICOM slices. Never infer
series boundaries from pixels or an anonymous concatenated corpus.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset


SLOTS = (("Sagittal", 1, 18), ("Sagittal", 0, 14),
         ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12))


@dataclass(frozen=True)
class TripletSpec:
    slot: int
    series_uid: str
    indices: tuple[int, int, int]


@dataclass(frozen=True)
class SamplingPlan:
    triplets: tuple[TripletSpec, ...]
    sampled_indices: tuple[tuple[int, ...], ...]
    target_triplets: int
    unique_triplets: int
    repeated_triplets: int

    @property
    def source_count(self) -> int:
        return sum(len(indices) for indices in self.sampled_indices)


def _allocate(capacities, weights, target):
    """Weighted largest-remainder allocation with capacity redistribution."""
    capacities = np.asarray(capacities, dtype=np.int64)
    weights = np.asarray(weights, dtype=np.float64)
    ideal = target * weights / weights.sum()
    quotas = np.floor(ideal).astype(np.int64)
    remainder = target - int(quotas.sum())
    quotas[np.argsort(-(ideal - quotas), kind="stable")[:remainder]] += 1
    quotas = np.minimum(quotas, capacities)
    wanted = min(target, int(capacities.sum()))
    deficit = np.zeros(len(quotas))
    while int(quotas.sum()) < wanted:
        available = quotas < capacities
        active_weights = np.where(available, weights, 0)
        deficit[available] += active_weights[available] / active_weights.sum()
        choice = int(np.argmax(np.where(available, deficit, -np.inf)))
        quotas[choice] += 1
        deficit[choice] -= 1
    return quotas


def plan_triplets(capacities, series_uids, weights=None, *,
                  target_triplets: int | None = None, legacy_slice_budget: int = 96,
                  shortage: str = "repeat") -> SamplingPlan:
    """Allocate real slices separately in each series and preserve the old K.

    Default K = min(96, total eligible source slices) - 2, matching dense
    parent inference. A series assigned q triplets needs q+2 source slices.
    If physical capacity is insufficient, repeat valid complete triplets
    (audited), or use shortage='error'. Never repeat a channel to fabricate
    a three-slice triplet and never join slices across series.
    """
    capacities = np.asarray(capacities)
    if (capacities.ndim != 1 or len(capacities) == 0
            or not np.issubdtype(capacities.dtype, np.integer)
            or np.any(capacities < 0) or len(series_uids) != len(capacities)):
        raise ValueError("Expected nonnegative integer capacity and UID per series slot")
    weights = np.ones(len(capacities)) if weights is None else np.asarray(weights, dtype=float)
    if weights.shape != capacities.shape or not np.isfinite(weights).all() or np.any(weights <= 0):
        raise ValueError("Expected positive finite allocation weights")
    if shortage not in ("repeat", "error"):
        raise ValueError("shortage must be 'repeat' or 'error'")
    present = [str(uid) for uid, n in zip(series_uids, capacities) if n > 0]
    if any(uid.strip().lower() in ("", "none", "nan", "<na>") for uid in present) or len(present) != len(set(present)):
        raise ValueError("Each populated slot must have its own known SeriesInstanceUID")
    if not isinstance(legacy_slice_budget, int) or legacy_slice_budget < 3:
        raise ValueError("legacy_slice_budget must be at least three")
    target = (min(legacy_slice_budget, int(capacities.sum())) - 2
              if target_triplets is None else target_triplets)
    if not isinstance(target, int) or isinstance(target, bool) or target < 1:
        raise ValueError("Triplet count must be a positive integer")
    available = np.maximum(capacities - 2, 0)
    if int(available.sum()) == 0:
        raise ValueError("No series contains three distinct acquired slices")
    if shortage == "error" and int(available.sum()) < target:
        raise ValueError(f"Need {target} unique same-series triplets; only {int(available.sum())} possible")
    quotas = _allocate(available, weights, target)
    sampled, specs = [], []
    for slot, (capacity, quota, uid) in enumerate(zip(capacities, quotas, series_uids)):
        indices = (np.linspace(0, int(capacity) - 1, int(quota) + 2).round().astype(int)
                   if quota else np.empty(0, dtype=int))
        if len(indices) != len(np.unique(indices)):
            raise RuntimeError("Source allocation selected duplicate slices")
        sampled.append(tuple(int(i) for i in indices))
        for start in range(int(quota)):
            specs.append(TripletSpec(slot, str(uid), tuple(int(i) for i in indices[start:start+3])))
    unique = len(specs)
    if unique < target:
        # Preserve every unique window, then fill the deficit with evenly spread
        # safe windows. The repetition count is returned for experiment audits.
        extra = np.linspace(0, unique - 1, target - unique).round().astype(int)
        specs.extend(specs[int(i)] for i in extra)
    return SamplingPlan(tuple(specs), tuple(sampled), target, unique, target - unique)


def _protocol_flag(value):
    try:
        number = float(value)
        return int(number) if np.isfinite(number) and number in (0, 1) else None
    except (ValueError, TypeError):
        text = str(value).strip().lower()
        return 1 if text in ("true", "yes", "y", "t") else 0 if text in ("false", "no", "n", "f") else None


def _pick_series(rows, plane, fluid, used):
    candidates = [row for row in rows if row["Anatomical_Plane"] == plane
                  and str(row["SeriesInstanceUID"]) not in used]
    preferred = [row for row in candidates if _protocol_flag(row.get("Fluid_Sensitive")) == fluid]
    return (preferred or candidates)[0] if candidates else None


def dicom_reader():
    """Order each individual series in patient coordinates and decode pixels."""
    import pydicom
    from pydicom.pixels import apply_modality_lut

    def order_and_meta(directory):
        directory = Path(directory)
        records = []
        for file in directory.glob("*.dcm"):
            header = pydicom.dcmread(file, stop_before_pixels=True)
            if str(header.SeriesInstanceUID) != directory.name:
                raise ValueError(f"DICOM SeriesInstanceUID does not match its directory: {file}")
            orientation = getattr(header, "ImageOrientationPatient", None)
            position = getattr(header, "ImagePositionPatient", None)
            if orientation is not None and position is not None:
                normal = np.cross(np.asarray(orientation[:3], float), np.asarray(orientation[3:], float))
                coordinate = float(np.dot(np.asarray(position, float), normal))
            else:
                coordinate = float(header.InstanceNumber)
            if not np.isfinite(coordinate):
                raise ValueError(f"Nonfinite slice coordinate: {file}")
            spacing = float(getattr(header, "PixelSpacing", [0.5])[0])
            records.append((coordinate, str(file), spacing, str(header.SOPInstanceUID)))
        records.sort(key=lambda record: (record[0], record[1]))
        if len({record[3] for record in records}) != len(records):
            raise ValueError(f"Duplicate acquired SOPInstanceUID in {directory}")
        median = float(np.median([record[2] for record in records])) if records else 0.5
        return [(file, spacing) for _, file, spacing, _ in records], median

    def read_px(path):
        dataset = pydicom.dcmread(path)
        image = apply_modality_lut(dataset.pixel_array, dataset).astype(np.float32)
        if image.ndim != 2 or not np.isfinite(image).all():
            raise ValueError(f"Invalid MRI slice pixels: {path}")
        if str(getattr(dataset, "PhotometricInterpretation", "")) == "MONOCHROME1":
            image = image.max() - image
        return image

    return order_and_meta, read_px


def prepare_study_triplets(study_uid: str, series_rows, series_root,
                           *, target_triplets=None, slots=SLOTS, span=(0.02, 0.98),
                           norm="imagenet", shortage="repeat", reader=None):
    """Raw-DICOM replacement for the parent's concatenated global stack.

    Return ([K,3,384,384], audit). Keep the parent five-slot selection, source
    depth band, series percentile reference, 140mm center crop and normalization.
    Every window comes from exactly one series, including shortage repeats.
    A reader=(order_and_meta, read_px) can reuse an existing DICOM cache.
    """
    import cv2

    if norm not in ("imagenet", "none"):
        raise ValueError("norm must be 'imagenet' or 'none'")
    if not 0 <= span[0] < span[1] <= 1:
        raise ValueError("Invalid source depth span")
    order_and_meta, read_px = dicom_reader() if reader is None else reader[:2]
    used, pools, capacities, uids = set(), [], [], []
    for plane, fluid, reference_count in slots:
        record = _pick_series(series_rows, plane, fluid, used)
        if record is None:
            pools.append(None); capacities.append(0); uids.append(None)
            continue
        uid = str(record["SeriesInstanceUID"])
        used.add(uid)
        files, spacing = order_and_meta(Path(series_root) / str(study_uid) / uid)
        lo = int(len(files) * span[0])
        hi = min(len(files)-1, max(int(len(files) * span[1])-1, lo))
        capacity = max(0, hi-lo+1)
        reference = np.linspace(lo, hi, reference_count).round().astype(int) if capacity else []
        pools.append((files, spacing, lo, reference))
        capacities.append(capacity); uids.append(uid)
    plan = plan_triplets(capacities, uids, [slot[2] for slot in slots],
                         target_triplets=target_triplets, shortage=shortage)
    planes, source_files, decoded_files = {}, [], []
    for slot, (pool, selected) in enumerate(zip(pools, plan.sampled_indices)):
        if pool is None or not selected:
            continue
        files, median_spacing, lo, reference = pool
        picks = [lo+i for i in selected]
        arrays = {i: read_px(files[i][0]) for i in sorted(set(picks) | set(reference))}
        decoded_files.extend(str(files[i][0]) for i in arrays)
        low, high = np.percentile(np.concatenate([arrays[int(i)].ravel() for i in reference]), [2, 98])
        for local, index in zip(selected, picks):
            path, spacing = files[index]
            image = np.clip((arrays[index]-low)/(high-low+1e-6), 0, 1)
            spacing = spacing if spacing > 0 else median_spacing
            h, w = image.shape
            size = min(int(round(140.0/max(spacing, .001))), min(h, w))
            y, x = (h-size)//2, (w-size)//2
            image = cv2.resize(image[y:y+size, x:x+size], (384, 384), interpolation=cv2.INTER_AREA)
            # Retain the parent's uint8 quantization before tensor conversion.
            planes[(slot, local)] = (image*255).astype(np.uint8)
            source_files.append(str(path))
    if len(source_files) != len(set(source_files)):
        raise ValueError("Selected source file occurs in multiple series slots")
    windows = np.stack([np.stack([planes[(spec.slot, index)] for index in spec.indices])
                        for spec in plan.triplets]).astype(np.float32) / 255.0
    tensor = torch.from_numpy(windows)
    if norm == "imagenet":
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        tensor = (tensor-mean)/std
    audit = {"study_uid": str(study_uid), "capacities": capacities,
             "triplets": len(plan.triplets), "unique_triplets": plan.unique_triplets,
             "repeated_triplets": plan.repeated_triplets,
             "sampled_slices": plan.source_count, "decoded_slices": len(set(decoded_files)),
             "series_uids": uids, "sampled_slices_per_series": [len(x) for x in plan.sampled_indices],
             "window_series_uids": [spec.series_uid for spec in plan.triplets],
             "window_source_files": [[str(pools[spec.slot][0][pools[spec.slot][2]+i][0])
                                      for i in spec.indices] for spec in plan.triplets]}
    return tensor, audit


class SeriesTripletDataset(Dataset):
    """Training K stays fixed; eval defaults to the parent's per-study dense K.

    `series` maps study UIDs to rows from train_series.csv/test_series.csv.
    Anonymous all_vols.npy stacks cannot enforce SeriesInstanceUID ownership;
    use raw series, or rebuild a cache retaining equivalent source metadata.
    """

    def __init__(self, ids, series, series_root, labels=None, *, train=False,
                 k=12, eval_k=None, norm="imagenet", aug=True, shortage="repeat"):
        self.ids, self.series, self.series_root, self.labels = ids, series, series_root, labels
        self.train, self.k, self.eval_k = train, k, eval_k
        self.norm, self.aug, self.shortage = norm, aug, shortage
        if not isinstance(k, int) or k < 1:
            raise ValueError("Training K must be positive")
        self.reader = None

    def __len__(self):
        return len(self.ids)

    def __getstate__(self):
        # DICOM reader closures are constructed inside each spawned worker.
        state = self.__dict__.copy()
        state["reader"] = None
        return state

    def load(self, index):
        uid = str(self.ids[index])
        if self.reader is None:
            self.reader = dicom_reader()
        # Plan the same dense candidate count as inference, then select K for
        # training. This keeps sparse studies' eval counts equal to the parent.
        x, audit = prepare_study_triplets(uid, self.series[uid], self.series_root,
                                          target_triplets=self.eval_k, norm="none",
                                          shortage=self.shortage, reader=self.reader)
        if self.train:
            first_positions = {}
            for i, paths in enumerate(audit["window_source_files"]):
                first_positions.setdefault(tuple(paths), i)
            candidates = torch.tensor(list(first_positions.values()))
            if self.shortage == "error" and self.k > len(candidates):
                raise ValueError("Insufficient training triplets without repetition")
            if self.k <= len(candidates):
                selected = candidates[torch.randperm(len(candidates))[:self.k]]
            else:
                selected = torch.cat([candidates[torch.randperm(len(candidates))]
                                      for _ in range((self.k+len(candidates)-1)//len(candidates))])[:self.k]
            x = x[selected]
            for key in ("window_series_uids", "window_source_files"):
                audit[key] = [audit[key][i] for i in selected.tolist()]
            audit["candidate_triplets"] = audit["triplets"]
            audit["triplets"] = self.k
            keys = [tuple(paths) for paths in audit["window_source_files"]]
            audit["unique_triplets"] = len(set(keys))
            audit["repeated_triplets"] = self.k-len(set(keys))
            if self.aug:
                x = (x * (0.9+0.2*torch.rand(()))).clamp(0, 1)
        if self.norm == "imagenet":
            x = (x-torch.tensor([.485, .456, .406]).view(1, 3, 1, 1)) / torch.tensor([.229, .224, .225]).view(1, 3, 1, 1)
        elif self.norm != "none":
            raise ValueError("norm must be 'imagenet' or 'none'")
        return x, audit

    def __getitem__(self, index):
        x, _ = self.load(index)
        return x if self.labels is None else (x, torch.as_tensor(self.labels[str(self.ids[index])], dtype=torch.float32))


In [ ]:
import hashlib

def attached_asset(relative_paths):
    existing = [Path(p) for p in relative_paths if Path(p).is_file()]
    if len(existing) != 1:
        raise FileNotFoundError(f'Expected one attached parent asset: {relative_paths}; found {existing}')
    return existing[0]

raptor_path = attached_asset([
    '/kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan/raptor_ft_coatnet_v5_full_swa.pt',
    '/kaggle/input/raptor-knee-maxspan/raptor_ft_coatnet_v5_full_swa.pt',
])
rad_path = attached_asset([
    '/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt',
    '/kaggle/input/resnet-50-radimagenet-marwan/ResNet50.pt',
])
digest = hashlib.sha256()
with rad_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
if digest.hexdigest() != '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734':
    raise RuntimeError('RadImageNet checkpoint differs from the baseline manifest')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = FusionConfig()
model = build_raptor_v5_radimagenet(raptor_path, rad_path, config).to(device).train()
optimizer = torch.optim.AdamW(optimizer_groups(model), weight_decay=0.01)
print('Parameters:', sum(p.numel() for p in model.parameters()))
print('Trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('Optimizer groups:', [(g['name'], g['lr']) for g in optimizer.param_groups])


In [ ]:
import pandas as pd

def make_series_loader(series_csv, series_root, ids, labels=None, *, train=False, k=12):
    # Preserve your existing train/validation study split and label mapping.
    frame = pd.read_csv(series_csv, dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    series = {uid: group.to_dict('records') for uid, group in frame.groupby('StudyInstanceUID')}
    dataset = SeriesTripletDataset(ids, series, series_root, labels,
                                  train=train, k=k, eval_k=None, norm='imagenet')
    # Eval K is the historical per-study count and can vary for sparse studies.
    # batch_size=1 handles that and bounds memory for two unfrozen backbones.
    return torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=train, num_workers=0)

# Example with your existing split IDs and label dict (12 targets per study):
# train_loader = make_series_loader(COMP_ROOT/'train_series.csv', COMP_ROOT/'train_images',
#                                   train_ids, labels, train=True, k=12)
# val_loader = make_series_loader(COMP_ROOT/'train_series.csv', COMP_ROOT/'train_images',
#                                 val_ids, labels, train=False)
# Each channel is a distinct acquired slice from one SeriesInstanceUID.


In [ ]:
# Use the series-aware loader above; it yields [B,K,3,384,384].
# Shared preprocessing/normalization is applied ONCE by that loader.
use_amp = device.type == 'cuda'
amp_dtype = (torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported()
             else torch.float16)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp and amp_dtype == torch.float16)

def train_step(triplets, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
        logits = model(triplets.to(device))
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets.to(device=device, dtype=torch.float32))
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach())

# After training, preserve both trunks, both fusion MLPs, and the original head:
# checkpoint_data = fused_checkpoint(model)
# checkpoint_data.update(optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), epoch=epoch)
# torch.save(checkpoint_data, '/kaggle/working/raptor_v5_radimagenet.pt')
